In [ ]:
"""
Script for interactive polygon drawing over Sentinel-2 RGB images for land cover classification.
Скрипт за интерактивно рисуване на полигони върху RGB изображения от Sentinel-2 за класификация на земното покритие.

It allows the user to manually delineate areas belonging to predefined land cover classes
(deciduous forest, coniferous, agriculture, bare lands, water) for a specific wildfire event.
Позволява на потребителя ръчно да очертае области, принадлежащи към предварително зададени класове
(широколистна гора, иглолистна, земеделска земя, голи площи, вода) за конкретен пожар.

Dependencies: rasterio, matplotlib, geopandas, shapely, numpy
Зависимости: rasterio, matplotlib, geopandas, shapely, numpy

Usage in Jupyter: %matplotlib qt
Използване в Jupyter: %matplotlib qt
"""

# ============================================================
# (Optional) Step 0 – Load fire metadata from CSV
# (Опционално) Стъпка 0 – Зареждане на метаданни за пожарите от CSV
# ============================================================
import pandas as pd
file_path = r'D:\data\master_thesis\input\fires_suggestion.csv'
df = pd.read_csv(file_path)
# Extract columns: Fire_id, Lat, Lon, Date
# Извличане на колони: Fire_id, Lat, Lon, Date
lat_col = 'Lat'
lon_col = 'Lon'
date_col = 'Date'
fire_id = 'Fire_id'
extracted_df = df[[fire_id, lat_col, lon_col, date_col]].dropna()
extracted_df[date_col] = pd.to_datetime(extracted_df[date_col], errors='coerce')
print(f"Loaded {len(extracted_df)} fire records.")
# ============================================================

# RUN THIS IN A JUPYTER CELL to enable interactive matplotlib window
# ИЗПЪЛНЕТЕ ТОВА В JUPYTER КЛЕТКА, за да активирате интерактивния прозорец на matplotlib
%matplotlib qt 

import rasterio
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Polygon
import os
import numpy as np
import re

# ==========================================
# HARDCODED PARAMETER: Change this number to process a different fire
# ТВЪРДО ЗАДАДЕН ПАРАМЕТЪР: Променете този номер, за да обработите друг пожар
# ==========================================
FIRE_NUMBER = 16
# ==========================================

# --- DIRECTORIES & PATHS ---
# --- ДИРЕКТОРИИ И ПЪТИЩА ---
# Base directory containing the Sentinel-2 classification images (.tif)
# Основна директория, съдържаща класификационните изображения от Sentinel-2 (.tif)
base_image_dir = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures'

# Subfolder where drawn polygons will be saved (one subfolder per fire)
# Подпапка, където ще се записват начертаните полигони (по една подпапка за всеки пожар)
base_poly_dir = os.path.join(base_image_dir, 'polygons')

# Sentinel-2 bands for an RGB composite: Band 4 (Red), 3 (Green), 2 (Blue)
# Канали на Sentinel-2 за RGB композит: Канал 4 (червен), 3 (зелен), 2 (син)
RGB_BANDS = [4, 3, 2] 

# Land cover classes to be delineated
# Класове земно покритие, които ще се очертават
classes = ['forest_deciduous', 'coniferous', 'field_agriculture', 'bare_lands', 'water']

def process_fire():
    """
    Main function that performs the interactive polygon drawing for the selected fire.
    Основна функция, която извършва интерактивното рисуване на полигони за избрания пожар.
    """
    # 1. FIND THE SPECIFIC IMAGE FILE
    # 1. НАМИРАНЕ НА КОНКРЕТНИЯ ФАЙЛ С ИЗОБРАЖЕНИЕТО
    all_files = [f for f in os.listdir(base_image_dir) if f.endswith('.tif')]
    # Regex pattern to match "fire16_" (for example) and avoid matching "fire160"
    # Регулярен израз, който съвпада с "fire16_" (например) и избягва съвпадение с "fire160"
    pattern = re.compile(f"fire{FIRE_NUMBER}[^0-9]") 
    target_files = [f for f in all_files if pattern.search(f)]

    if not target_files:
        print(f"!!! Error: Could not find image for Fire {FIRE_NUMBER} in {base_image_dir}")
        return

    filename = target_files[0]
    fire_id = f"fire{FIRE_NUMBER}"
    fire_folder_name = f"fire_{FIRE_NUMBER}"
    fire_output_path = os.path.join(base_poly_dir, fire_folder_name)
    os.makedirs(fire_output_path, exist_ok=True)   # Create output folder if it doesn't exist
                                                   # Създаване на изходна папка, ако не съществува
    
    full_image_path = os.path.join(base_image_dir, filename)
    print(f"--- ACTIVE SESSION: {fire_id.upper()} ---")

    with rasterio.open(full_image_path) as src:
        # 2. PREPARE THE RGB IMAGE FOR DISPLAY
        # 2. ПОДГОТОВКА НА RGB ИЗОБРАЖЕНИЕТО ЗА ВИЗУАЛИЗАЦИЯ
        try:
            # Read the predefined Sentinel-2 bands
            # Прочитане на предварително зададените канали на Sentinel-2
            img_data = src.read(RGB_BANDS)
        except:
            # Fallback: if the file has a different band order, use first three bands
            # Резервен вариант: ако файлът има различен ред на каналите, използвай първите три
            img_data = src.read([1, 2, 3])
        
        # Transpose to (height, width, channels) and convert to float for contrast stretching
        # Транспониране до (височина, ширина, канали) и преобразуване в float за разтягане на контраста
        img_display = np.transpose(img_data, (1, 2, 0)).astype(float)
        for i in range(3):
            p5, p95 = np.percentile(img_display[:,:,i], (5, 95))   # 5th and 95th percentiles
                                                                   # 5-ти и 95-ти персентил
            img_display[:,:,i] = np.clip((img_display[:,:,i] - p5) / (p95 - p5), 0, 1)

        # 3. INTERACTIVE POLYGON DRAWING FOR EACH LAND COVER CLASS
        # 3. ИНТЕРАКТИВНО РИСУВАНЕ НА ПОЛИГОНИ ЗА ВСЕКИ КЛАС ЗЕМНО ПОКРИТИЕ
        for land_type in classes:
            # Create a new figure for each class
            # Създаване на нова фигура за всеки клас
            fig, ax = plt.subplots(figsize=(10, 10))
            ax.imshow(img_display)
            ax.set_title(f"FIRE {FIRE_NUMBER} | CLASS: {land_type.upper()}\nL-Click: Points | R-Click: Finish Poly\nCLOSE WINDOW: Skip/Move to Next Class")
            
            all_class_geometries = []   # List to hold completed polygons (shapely Polygons)
                                        # Списък за съхранение на завършените полигони (shapely Polygons)
            current_poly_pts = []       # Temporary list for points of the polygon currently being drawn
                                        # Временен списък за точките на полигона, който се рисува в момента
            
            def onclick(event):
                """
                Mouse click event handler.
                Обработчик на събития при кликване с мишката.

                Left click: adds a vertex to the current polygon.
                Ляв бутон: добавя връх към текущия полигон.

                Right click: closes the polygon if at least 3 points exist, stores it,
                and clears the temporary list for a new polygon.
                Десен бутон: затваря полигона, ако има поне 3 точки, запазва го
                и изчиства временния списък за нов полигон.
                """
                nonlocal current_poly_pts
                if event.button == 1: # Left Click / Ляв клик
                    current_poly_pts.append((event.xdata, event.ydata))
                    x, y = zip(*current_poly_pts)
                    ax.plot(x, y, color='yellow', marker='o', markersize=3)  # draw points and lines
                                                                             # рисуване на точки и линии
                    if len(current_poly_pts) > 1:
                        ax.plot([current_poly_pts[-2][0], current_poly_pts[-1][0]], 
                                [current_poly_pts[-2][1], current_poly_pts[-1][1]], color='yellow')
                    fig.canvas.draw()   # update the figure
                                       # обновяване на фигурата
                
                elif event.button == 3 and len(current_poly_pts) > 2: # Right Click & enough points
                                                                       # Десен клик & достатъчно точки
                    # Draw closing edge in cyan for visual feedback
                    # Рисуване на затварящото ребро в циан за визуална обратна връзка
                    ax.plot([current_poly_pts[-1][0], current_poly_pts[0][0]], 
                            [current_poly_pts[-1][1], current_poly_pts[0][1]], color='cyan', lw=2)
                    
                    # Convert pixel coordinates to geographic coordinates using the raster's transform
                    # Преобразуване на пикселни координати в географски чрез трансформацията на растера
                    map_pts = [src.xy(int(y), int(x)) for x, y in current_poly_pts]
                    all_class_geometries.append(Polygon(map_pts))   # Store the closed polygon
                                                                   # Запазване на затворения полигон
                    current_poly_pts = []  # Reset for the next polygon
                                          # Нулиране за следващия полигон
                    fig.canvas.draw()

            # Connect the mouse click event to the handler
            # Свързване на събитието кликване на мишката с обработчика
            fig.canvas.mpl_connect('button_press_event', onclick)
            
            # Show the figure and block until the window is closed
            # Показване на фигурата и блокиране до затваряне на прозореца
            plt.show(block=True) 

            # 4. SAVE THE DRAWN POLYGONS AS A GEOPACKAGE (if any were created)
            # 4. ЗАПАЗВАНЕ НА НАЧЕРТАНИТЕ ПОЛИГОНИ КАТО GEOPACKAGE (ако има създадени)
            if all_class_geometries:
                # Create a GeoDataFrame with a 'class' attribute and the geometries
                # Създаване на GeoDataFrame с атрибут 'class' и геометриите
                gdf = gpd.GeoDataFrame({'class': [land_type]*len(all_class_geometries)}, 
                                       geometry=all_class_geometries, crs=src.crs)
                
                save_name = f"{fire_id}_{land_type}.gpkg"
                gdf.to_file(os.path.join(fire_output_path, save_name), driver="GPKG")
                print(f"   -> Saved {len(all_class_geometries)} polygons for {land_type}")
            else:
                print(f"   -> No polygons created for {land_type}. Skipping file creation.")
            
            plt.close(fig)  # Ensure the figure is closed before moving to the next class
                           # Уверяване, че фигурата е затворена преди преминаване към следващия клас

    print(f"\nCompleted Fire {FIRE_NUMBER}. Files saved in: {fire_output_path}")
    print(f"\nЗавършен пожар {FIRE_NUMBER}. Файловете са запазени в: {fire_output_path}")

if __name__ == "__main__":
    process_fire()